# 🚀 CVAT Universal Multi-Modal AI Auto-Annotation (Google Colab 1-Click Runner)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanlee199/locate-anything/blob/locateV2/colab/launch_colab.ipynb)

Hệ thống AI tự động hóa gán nhãn đa hình thái trên **Google Colab (GPU T4 miễn phí)** kết nối trực tiếp với **CVAT Server**:
- 🧊 **3D Point Cloud LiDAR (Cuboid)**: Tự động phát hiện và vẽ hộp lập phương 3D cho file `.pcd` / `.bin` bằng mô hình **nuScenes PointPillars Deep Learning** (10 classes: `car`, `truck`, `bus`, `trailer`, `construction_vehicle`, `pedestrian`, `motorcycle`, `bicycle`, `barrier`, `traffic_cone`).
- 🎭 **Native Bitmap Mask (Brush RLE)**: Tự động phân đoạn điểm ảnh chính xác chuẩn CVAT RLE (SAM 2.1).
- 📐 **Vector Polygon**: Tự động sinh đa giác viền khít có điểm neo (YOLO11 Segmenter).
- 📦 **2D Bounding Box**: Nhận diện xe cộ, người đi bộ, chướng ngại vật (YOLO11 Detector).
- 📏 **Polyline / Line**: Rút xương dải đường thành đường tim đường.
- 🦴 **Skeleton / Pose**: Nhận diện khung xương người (17 keypoints).
- ⭕ **Ellipse**: Tự động khớp phương trình elip toán học.
- 🏷️ **Tag**: Phân loại toàn ảnh tự động.

---
### ⚡ Hướng dẫn sử dụng nhanh (Chỉ 2 bước):
1. **Bật GPU**: Menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU** $\rightarrow$ Bấm **Save**.
2. **Chạy Bước 1**: Bấm nút Play ở **BƯỚC 1** để tự động clone code và thiết lập môi trường AI.
3. **Chạy Bước 2**: Nhập URL CVAT + Token của bạn vào Form và bấm Play ở **BƯỚC 2** để AI tự động gán nhãn!

In [ ]:
#@title 📦 BƯỚC 1: Tự Động Thiết Lập Môi Trường GPU & Mã Nguồn { display-mode: "form" }
#@markdown Bấm nút Play cell này để tự động cập nhật code mới nhất từ GitHub, cấu hình GPU và nạp toàn bộ trọng số AI (2D & 3D nuScenes).

import os
import sys
import subprocess

print("=" * 70)
print("🚀 [BƯỚC 1/2] THIẾT LẬP HỆ THỐNG CVAT AI AUTO-ANNOTATION TRÊN COLAB")
print("=" * 70)

# 1. Kiểm tra GPU
try:
    import torch
    if not torch.cuda.is_available():
        print("⚠️ CẢNH BÁO: Chưa kích hoạt GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU -> Bấm Save rồi chạy lại cell này.")
    else:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"✅ Phát hiện GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
except Exception as e:
    print(f"⚠️ Không kiểm tra được GPU: {e}")

# 2. Tự động Clone hoặc Pull nhánh locateV2 mới nhất
REPO_DIR = "/content/locate-anything"
BRANCH = "main"
REPO_URL = "https://github.com/Wanlee199/locate-anything.git"

if os.path.exists(REPO_DIR):
    print(f"🔄 Đang cập nhật mã nguồn mới nhất từ nhánh {BRANCH}...")
    subprocess.run(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}", shell=True, check=True)
else:
    print(f"📥 Đang tải mã nguồn từ {REPO_URL} (nhánh {BRANCH})...")
    subprocess.run(f"git clone -b {BRANCH} {REPO_URL} {REPO_DIR}", shell=True, check=True)

# 3. Chuyển thư mục làm việc và đưa vào sys.path
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 4. Cài đặt các thư viện cơ bản từ requirements.txt
print("📦 Đang cài đặt các thư viện phụ thuộc từ requirements.txt...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt gdown", shell=True, check=True)

# 5. Cài đặt và tải mô hình 3D nuScenes PointPillars
print("🧊 Đang kiểm tra và thiết lập mô hình 3D nuScenes PointPillars...")
subprocess.run(f"{sys.executable} tools/setup_3d_model.py --install", shell=True, check=True)

print("\n" + "=" * 70)
print("🎉 HOÀN TẤT THIẾT LẬP MÔI TRƯỜNG! BÂY GIỜ HÃY CHẠY BƯỚC 2 BÊN DƯỚI 👇")
print("=" * 70)


🚀 [BƯỚC 1/2] THIẾT LẬP HỆ THỐNG CVAT AI AUTO-ANNOTATION TRÊN COLAB
✅ Phát hiện GPU: Tesla T4 (14.6 GB VRAM)
🔄 Đang cập nhật mã nguồn mới nhất từ nhánh locateV2...
📦 Đang cài đặt các thư viện phụ thuộc từ requirements.txt...


In [ ]:
#@title 🚀 BƯỚC 2: Cấu Hình & Chạy AI Tự Động Gán Nhãn Lên CVAT { display-mode: "form" }
#@markdown Điền thông tin kết nối CVAT Server và bấm nút Play để AI tự động phân tích và gán nhãn:

# URL máy chủ CVAT (Cloudflare Tunnel nếu CVAT trên máy cá nhân, hoặc IP VPS, hoặc https://app.cvat.ai)
CVAT_HOST = "https://filme-casting-charlotte-machine.trycloudflare.com" #@param {type:"string"}

# API Token lấy từ CVAT Web -> Bấm vào Avatar góc trên bên phải -> Profile -> API Tokens -> Create
CVAT_TOKEN = "EV9rAIxB.qO5g8FqupPEKx4xzOULG5NVoMz73Lhdd" #@param {type:"string"}

# ID của Job bạn được phân công (Ví dụ: 2). Nếu muốn chạy cả Task, hãy đặt Job ID = 0 và nhập Task ID bên dưới.
CVAT_JOB_ID = 80 #@param {type:"integer"}

# ID của Task (Ví dụ: 2). Nếu đã điền CVAT_JOB_ID thì hệ thống sẽ tự động nhận diện Task ID.
CVAT_TASK_ID = 54 #@param {type:"integer"}

# Độ nhạy phát hiện vật thể (0.1 = nhạy nhất phát hiện nhiều, 0.5 = chặt chẽ độ chính xác cao)
CONFIDENCE_THRESHOLD = 0.5 #@param {type:"slider", min:0.05, max:0.95, step:0.05}

# Xóa các nhãn cũ/rác trên Job trước khi nạp nhãn AI mới (khuyên dùng True để tránh trùng lặp)
CLEAR_OLD_ANNOTATIONS = True #@param {type:"boolean"}

# Giới hạn số frame chạy thử (Đặt 0 để chạy TOÀN BỘ tất cả frames trong Job/Task)
MAX_FRAMES_TEST = 0 #@param {type:"integer"}

import os
import sys
import subprocess

# Đảm bảo đang đứng đúng thư mục repo
if not os.path.exists("colab/cvat_auto_sync.py"):
    if os.path.exists("/content/locate-anything"):
        os.chdir("/content/locate-anything")
    else:
        raise RuntimeError("Vui lòng chạy BƯỚC 1 trước để tải mã nguồn!")

# Xây dựng lệnh thực thi
cmd = [
    sys.executable, "colab/cvat_auto_sync.py",
    "--host", CVAT_HOST.strip(),
    "--token", CVAT_TOKEN.strip(),
    "--confidence", str(CONFIDENCE_THRESHOLD),
]

if CVAT_JOB_ID and CVAT_JOB_ID > 0:
    cmd.extend(["--job-id", str(CVAT_JOB_ID)])
if CVAT_TASK_ID and CVAT_TASK_ID > 0:
    cmd.extend(["--task-id", str(CVAT_TASK_ID)])
if not CLEAR_OLD_ANNOTATIONS:
    cmd.append("--keep-existing")
if MAX_FRAMES_TEST and MAX_FRAMES_TEST > 0:
    cmd.extend(["--max-frames", str(MAX_FRAMES_TEST)])

res = subprocess.run(cmd)
if res.returncode != 0:
    print(f"\n❌ Tiến trình gán nhãn gặp lỗi (Exit code {res.returncode})!")


In [ ]:
#@title 🩺 BƯỚC 3 (Tùy chọn): Kiểm Tra Kết Nối & Thông Tin Task / Job { display-mode: "form" }
#@markdown Cell này kiểm tra nhanh kết nối tới CVAT, danh sách nhãn và định dạng dữ liệu (2D hay 3D LiDAR):

import json
import urllib.request
import urllib.error

headers = {
    "Authorization": f"Bearer {CVAT_TOKEN.strip()}",
    "Accept": "application/vnd.cvat+json, application/json;q=0.9",
}

base_url = CVAT_HOST.strip().rstrip("/")
print(f"🔍 Đang kiểm tra kết nối tới: {base_url} ...")

try:
    # 1. Kiểm tra Job nếu có
    target_task_id = CVAT_TASK_ID
    if CVAT_JOB_ID and CVAT_JOB_ID > 0:
        job_req = urllib.request.Request(f"{base_url}/api/jobs/{CVAT_JOB_ID}", headers=headers)
        with urllib.request.urlopen(job_req) as resp:
            job_data = json.loads(resp.read().decode("utf-8"))
            target_task_id = job_data.get("task_id", target_task_id)
            print(f"✅ Kết nối thành công Job #{CVAT_JOB_ID}!")
            print(f"   • Trạng thái Job: {job_data.get('state')} / {job_data.get('stage')}")
            print(f"   • Khung hình: Frame {job_data.get('start_frame')} -> {job_data.get('stop_frame')} (Tổng: {job_data.get('stop_frame') - job_data.get('start_frame') + 1} frames)")
            print(f"   • Thuộc Task ID: #{target_task_id}")

    # 2. Kiểm tra Task
    if target_task_id and target_task_id > 0:
        task_req = urllib.request.Request(f"{base_url}/api/tasks/{target_task_id}", headers=headers)
        with urllib.request.urlopen(task_req) as resp:
            task_data = json.loads(resp.read().decode("utf-8"))
            dim = task_data.get("dimension", "2d")
            print(f"\n📋 Thông tin Task #{target_task_id}: {task_data.get('name')}")
            print(f"   • Kiểu dữ liệu: {'🧊 3D Point Cloud LiDAR (.pcd)' if dim == '3d' else '🖼️ 2D Images / Video'}")
            print(f"   • Tổng số frames của Task: {task_data.get('size')} frames")

        # 3. Kiểm tra nhãn
        lbl_req = urllib.request.Request(f"{base_url}/api/labels?task_id={target_task_id}", headers=headers)
        with urllib.request.urlopen(lbl_req) as resp:
            lbl_data = json.loads(resp.read().decode("utf-8"))
            labels = lbl_data.get("results", [])
            print(f"\n🏷️ Danh sách nhãn trong Task ({len(labels)} nhãn):")
            for l in labels:
                print(f"   • [ID: {l.get('id')}] {l.get('name')} (Type: {l.get('type')})")

    print("\n🟢 Kết nối và cấu hình hoàn toàn chuẩn xác!")
except Exception as e:
    print(f"\n❌ Lỗi kiểm tra: {e}")
